In [1]:
# Pegar o arquivo .csv direto do google drive
from google.colab import drive
drive.mount('/content/drive')

# Bibliotecas para a análise bruta de dados
import pandas as pd

url = 'https://drive.google.com/uc?id=1EzS6B2UC1PWKUjyxqUGneCP4cYEa9ob1'
df = pd.read_csv(url, sep=',')
df.head()

Mounted at /content/drive


,ano,rede,ensino,anos_escolares,taxa_aprovacao,indicador_rendimento,nota_saeb_matematica,nota_saeb_lingua_portuguesa,nota_saeb_media_padronizada,ideb,projecao
0,2009,total,medio,todos (1-4),75.9,0.795302,274.72,268.82,4.572342,3.6,3.5
1,2013,total,medio,todos (1-4),80.1,0.823323,270.14,264.05,4.436750,3.7,3.9
2,2021,total,medio,todos (1-4),90.8,0.900923,271.00,275.97,4.626981,4.2,NaN
3,2007,total,medio,todos (1-4),74.1,0.778403,272.89,261.39,4.435226,3.5,3.4
4,2017,total,medio,todos (1-4),83.1,0.838888,270.63,268.51,4.510258,3.8,4.7


In [2]:
# Visão geral de dados

#tipos de dados
#colunas problemáticas
#possíveis NaN

df.info()
print(df.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ano                          126 non-null    int64  
 1   rede                         126 non-null    object 
 2   ensino                       126 non-null    object 
 3   anos_escolares               126 non-null    object 
 4   taxa_aprovacao               126 non-null    float64
 5   indicador_rendimento         126 non-null    float64
 6   nota_saeb_matematica         126 non-null    float64
 7   nota_saeb_lingua_portuguesa  126 non-null    float64
 8   nota_saeb_media_padronizada  126 non-null    float64
 9   ideb                         126 non-null    float64
 10  projecao                     98 non-null     float64
dtypes: float64(7), int64(1), object(3)
memory usage: 11.0+ KB
(126, 11)


In [3]:
# Mostrar os Valores Nulos e deletar
df.isnull().sum().sort_values(ascending=False)

df = df.dropna()

df.isnull().sum().sort_values(ascending=False)

# Se tiver muitos: df.fillna(0, inplace=True)  # ou média depois

,0
ano,0
rede,0
ensino,0
anos_escolares,0
taxa_aprovacao,0
indicador_rendimento,0
nota_saeb_matematica,0
nota_saeb_lingua_portuguesa,0
nota_saeb_media_padronizada,0
ideb,0


In [4]:
# Duplicados
df.duplicated().sum()

np.int64(0)

In [5]:
# Se tiver
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [6]:
# Converter colunas numéricas
cols_numericas = [
    'taxa_aprovacao',
    'indicador_rendimento',
    'nota_saeb_matematica',
    'nota_saeb_lingua_portuguesa'
]

for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [7]:
# Garantir coluna "ano" como inteiro
df['ano'] = df['ano'].astype(int)

In [8]:
# Padronização de texto
df['rede'] = df['rede'].str.lower().str.strip()
df['ensino'] = df['ensino'].str.lower().str.strip()

In [9]:
# Valores unicos (sujeira escondida)
df['ensino'].unique()


array(['medio', 'fundamental'], dtype=object)

In [10]:
df['anos_escolares'].unique()

array(['todos (1-4)', 'finais (6-9)', 'iniciais (1-5)'], dtype=object)

In [11]:
# Coluna anterior (anos_escolares) está mal estruturada, pois o valor está misturado em 1 só
# Solução: Separar as colunas em 2 diferentes

df['categoria'] = df['anos_escolares'].str.extract(r'^(.*?)\s')
# Começo da String, pega tudo antes de encontrar um espaço

In [12]:
df['faixa'] = df['anos_escolares'].str.extract(r'\((.*?)\)')
# Pega tudo entre parenteses

In [13]:
# Separar o ano de inicio e o ano de fim
df[['ano_inicio', 'ano_fim']] = df['faixa'].str.split('-', expand=True)

In [14]:
df['ano_inicio'] = df['ano_inicio'].astype(int)
df['ano_fim'] = df['ano_fim'].astype(int)

In [15]:
# Após todas essas mudanças, finalmente podemos acompanhar tudo por grupos ou etapas

In [16]:
# Desempenho por etapa
df.groupby('categoria')['taxa_aprovacao'].mean()

,taxa_aprovacao
categoria,
finais,85.825714
iniciais,92.620000
todos,82.410714


In [17]:
# Evolução por faixa etária
df.groupby(['ano', 'categoria'])['nota_saeb_matematica'].mean().unstack()

categoria,finais,iniciais,todos
ano,,,
2007,252.282,198.662,282.2425
2009,253.290,210.610,283.8450
2011,256.996,214.782,284.5075
2013,255.116,216.572,278.3525
2015,260.818,223.262,274.3375
2017,263.682,228.482,280.1325
2019,267.766,231.706,287.7150


In [18]:
# Com tudo confirmado e averiguado, podemos remover a coluna original
df = df.drop(columns=['anos_escolares'])

In [20]:
df.head()

,ano,rede,ensino,taxa_aprovacao,indicador_rendimento,nota_saeb_matematica,nota_saeb_lingua_portuguesa,nota_saeb_media_padronizada,ideb,projecao,categoria,faixa,ano_inicio,ano_fim
0,2009,total,medio,75.9,0.795302,274.72,268.82,4.572342,3.6,3.5,todos,1-4,1,4
1,2013,total,medio,80.1,0.823323,270.14,264.05,4.436750,3.7,3.9,todos,1-4,1,4
3,2007,total,medio,74.1,0.778403,272.89,261.39,4.435226,3.5,3.4,todos,1-4,1,4
4,2017,total,medio,83.1,0.838888,270.63,268.51,4.510258,3.8,4.7,todos,1-4,1,4
5,2011,total,medio,77.4,0.802240,274.82,268.57,4.569995,3.7,3.7,todos,1-4,1,4


In [26]:
# Evoluão da taxa de aprovação ao longo do tempo
df_ano = df.groupby('ano')['taxa_aprovacao'].mean().reset_index()
df_ano.head()

,ano,taxa_aprovacao
0,2007,82.857143
1,2009,84.414286
2,2011,86.121429
3,2013,87.757143
4,2015,88.485714


In [30]:
# Ensino fundamental VS Médio
df_ensino = df.groupby('ensino')['taxa_aprovacao'].mean().reset_index()
df_ensino.head()

,ensino,taxa_aprovacao
0,fundamental,89.222857
1,medio,82.410714


In [32]:
# Desempenho de Matemática VS Português
df_notas = df[['nota_saeb_matematica', 'nota_saeb_lingua_portuguesa']].mean().reset_index()
df_notas.columns = ['disciplina', 'nota']
df_notas.head()

,disciplina,nota
0,nota_saeb_matematica,250.557755
1,nota_saeb_lingua_portuguesa,241.170000


In [36]:
# Hora de automatizar o envio dos relatórios via email
# Será feito um usando Selenium e outro usando SMTP

# Qual a diferença entre os 2?

# O Selenium automatiza ações em navegadores web (como cliques e testes de páginas)

# SMTP (Simple Mail Transfer Protocol) é um protocolo de rede usado estritamente para enviar e
# transmitir mensagens de e-mail entre servidores.
# Eles servem para coisas totalmente diferentes na tecnologia.

In [34]:
# Selenium:
# Função: Automatiza navegadores web para testar sistemas ou interagir com telas de sites.
# Atua: Na camada de interface de usuário (UI) de uma aplicação web.
# Casos de uso: Preencher formulários automaticamente, testar se um botão funciona ou raspar dados de uma página web.

# Problemas com o Selenium:
# Gmail muda layout → quebra seu código
# Precisa login manual (ou cookies)
# Lento
# Não é escalável

In [37]:
# SMTP:
# Função: Transmite e entrega mensagens de e-mail pela internet.
# Atua: Na camada de rede e servidores de comunicação de correio eletrônico.
# Casos de uso: Enviar um alerta de sistema, um recibo de compra ou uma
# recuperação de senha para a caixa de entrada de um usuário.

# Problemas com o SMTP
# Segurança -> smtp.login('email', 'senha'), se fizer isso errado pode vazar credenciais ou expor em locais indevidos
# Bloqueios do provedor: Gmail pode bloquear “atividade suspeita”, envio automatizado ou muitos emails em sequência.
# Limites de envio, exemplo: ~500 emails/dia (conta normal) -> Passou disso → bloqueio.
# Configuração de entrada: Porta, SSL vs TLS ou autenticação. Pequeno erro = não funciona.
# Pode cair em SPAM sem a formatação adequada ou sem domínio o confiável, nesses casos seu email nem chega na caixa principal.
# Sem “interface visual”. Diferente do Selenium: não mostra o processo e também não interage com UI.

In [39]:
# Comparação final

#| Critério       | Selenium | SMTP  |
#| -------------- | ---------| ------|
#| Confiabilidade | baixa    | alta  |
#| Performance    | lenta    | rápida|
#| Escalabilidade | ruim     | média |
#| Facilidade     | média    | média |
#| Impacto visual | alto     | baixo |
#| Produção real  | não      | sim   |

In [46]:
# Automação usando o Selenium (08/2026)

!pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 12.8 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Caminho do seu perfil do Chrome (AJUSTE AQUI)
options = Options()
options.add_argument(r"user-data-dir=C:\Users\SEU_USUARIO\AppData\Local\Google\Chrome\User Data")
options.add_argument("profile-directory=Default")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

# Abrir Gmail
driver.get("https://mail.google.com/mail/u/0/#inbox")

# Botão "Escrever"
compose = wait.until(EC.element_to_be_clickable(
    (By.XPATH, "//div[contains(text(),'Escrever') or contains(text(),'Compose')]")
))
compose.click()

# Campo destinatário
to_field = wait.until(EC.presence_of_element_located((By.NAME, "to")))
to_field.send_keys("email_destino@gmail.com")

# Assunto
subject = driver.find_element(By.NAME, "subjectbox")
subject.send_keys("Relatório Automatizado")

# Corpo do email
body = wait.until(EC.presence_of_element_located(
    (By.XPATH, "//div[@aria-label='Corpo da mensagem' or @aria-label='Message Body']")
))
body.send_keys("Segue relatório gerado automaticamente.")

# Upload de arquivo
upload_input = wait.until(EC.presence_of_element_located(
    (By.XPATH, "//input[@type='file']")
))
upload_input.send_keys(r"C:\caminho\completo\relatorio.html")

# Botão enviar
send_button = wait.until(EC.element_to_be_clickable(
    (By.XPATH, "//div[@aria-label[contains(.,'Enviar')] or @aria-label[contains(.,'Send')]]")
))
send_button.click()

print("Email enviado com sucesso!")

driver.quit()

In [53]:
# Antes de usar o SMTP, precisa de uma preparação importante.
# SMTP — versão correta (Gmail)
# Você NÃO pode usar sua senha normal.
# Crie uma senha de app: https://myaccount.google.com/apppasswords

In [ ]:
import smtplib
from email.message import EmailMessage
from pathlib import Path

def enviar_email():
    # Configurações
    EMAIL = "seu_email@gmail.com"
    SENHA = "senha_de_app"
    DESTINATARIOS = ["email1@gmail.com", "email2@gmail.com"]

    # Criar mensagem
    msg = EmailMessage()
    msg["Subject"] = "Relatório Automatizado"
    msg["From"] = EMAIL
    msg["To"] = ", ".join(DESTINATARIOS)

    msg.set_content("Segue o relatório gerado automaticamente.")

    # Anexar arquivo
    caminho_arquivo = Path("reports/relatorio.html")

    with open(caminho_arquivo, "rb") as f:
        msg.add_attachment(
            f.read(),
            maintype="text",
            subtype="html",
            filename="relatorio.html"
        )

    # Enviar email
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
        smtp.login(EMAIL, SENHA)
        smtp.send_message(msg)

    print("Email enviado com sucesso!")

# Executar
enviar_email()